### ADLS End Point, Access Key and Container Name

In [0]:
storageAccountEndPoint = "dsdmqa1adls.dfs.core.windows.net"
storageAccountAccessKey = dbutils.secrets.get(scope = "ETP_DS_DB_SECRET_SCOPE", key = "storageAccessKey")
containerName = "mongocollections"

### ADLS config setting :: Storage Key to get direct access on ADLS

In [0]:
spark.conf.set(
  "fs.azure.account.key." + storageAccountEndPoint,
  storageAccountAccessKey)

### Storage Account Path

In [0]:
containerPath = "abfss://" + containerName + "@" + storageAccountEndPoint

## imports and declarations

In [0]:
from datetime import timedelta, datetime
import dateutil
import base64
from pyspark.sql.functions import col,row_number, date_format, concat, lit, concat_ws
from pyspark.sql.types import *
from pyspark.sql import Window
import pyspark.sql.functions as func
from pyspark.sql import DataFrame

### Set input Widgets

In [0]:
tenants = ["SOLAR","QA","QA1", "MSXMONIT"]
years = ["2014","2015","2016","2017","2018","2019","2020","2021","2022"]
metadataCollections = ["series","events"]
telemetryCollections = ["seriesEntry","eventsIntervalEntry","eventsTimestampedEntry"]
tags = ["series_id","series_id,source_id,instance_id"]
fields = ["attr_,data","attr_,data,end_time"]

# Create simple dropdown widgets
dbutils.widgets.dropdown("inputTenant", "QA1", tenants)
dbutils.widgets.dropdown("inputYear", "2019", years)
dbutils.widgets.dropdown("metadataCollection", "series", metadataCollections)
dbutils.widgets.dropdown("telemetryCollection", "seriesEntry", telemetryCollections)
dbutils.widgets.dropdown("tags", "series_id", tags)
dbutils.widgets.dropdown("fields", "attr_,data", fields)

### Get the input parameters

In [0]:
# Current value of the widget:
inputTenant = dbutils.widgets.get("inputTenant")
inputYear = dbutils.widgets.get("inputYear")
metadataCollection = dbutils.widgets.get("metadataCollection")
telemetryCollection = dbutils.widgets.get("telemetryCollection")
tags = dbutils.widgets.get("tags")
fields = dbutils.widgets.get("fields")

### ADLS storage Paths

In [0]:
# ADLS path for tenant collection's data
tenantPath = containerPath + "/Tenants/" + inputTenant + "/tenants.json" 

# ADLS path for metadata collection's data
metadataPath = containerPath + "/Tenants/" + inputTenant + "/" + metadataCollection + ".json" 

# ADLS path for telemetry collection's data
telemetryEntryPath = containerPath + "/Tenants/" + inputTenant + "/" + telemetryCollection+ "/" + inputYear + "/*/*/*.json"

# ADLS path to store line protocol
lineProtocolPath = containerPath + "/Tenants/" + inputTenant + "/" + "lineProtocol/" + telemetryCollection+ "/" + inputYear 
print(tenantPath)
print(lineProtocolPath)
print(telemetryEntryPath)

abfss://mongocollections@dsdmqa1adls.dfs.core.windows.net/Tenants/QA/tenants.json
abfss://mongocollections@dsdmqa1adls.dfs.core.windows.net/Tenants/QA/lineProtocolPerfImproved/seriesEntry/2020
abfss://mongocollections@dsdmqa1adls.dfs.core.windows.net/Tenants/QA/seriesEntry/2020/*/*/*.json

### Read tenant's data

In [0]:
tenant = spark.read.json(tenantPath)
storages = tenant.select("Storages").rdd.flatMap(lambda x: x).collect()
tenant.show()
print(len(storages[0]))

+----+--------------------+--------------------+
Name| Storages| _id|
+----+--------------------+--------------------+
 QA|[{main}, {storage...|{5EyZO2lafUCJ9h7g...|
+----+--------------------+--------------------+

3

### Series : Metadata device IDs read

In [0]:
seriesTest1 = spark.read.json(metadataPath)

seiesTestSelect1 = seriesTest1.select("_id.*","Storage","_t").createOrReplaceTempView("series_temp_table")
spark.sql("select count(*) from (select distinct Storage from series_temp_table)").show(truncate=False)

spark.sql("select * from series_temp_table where _t[2] = 'NumericTimeSeries'").createOrReplaceTempView("series_temp_table_numeric_series")
spark.sql("select * from series_temp_table where _t[2] = 'StringTimeSeries'").createOrReplaceTempView("series_temp_table_string_series")
spark.sql("select * from series_temp_table_numeric_series").show(truncate=False)
spark.sql("select * from series_temp_table_string_series").show(truncate=False)



+--------+
count(1)|
+--------+
2 |
+--------+

+------------------------+-----+--------+------------------------------------------+
$binary |$type|Storage |_t |
+------------------------+-----+--------+------------------------------------------+
EgCYT+PgQpG5BWfqPcDOFA==|03 |main |[DflObject, TimeSeries, NumericTimeSeries]|
HNo0K35780uzVuihKuk2Ig==|03 |main |[DflObject, TimeSeries, NumericTimeSeries]|
HuTiQwDFl0qusF18HMi6Fw==|03 |main |[DflObject, TimeSeries, NumericTimeSeries]|
NY+0f0FGTtK1MGPN6ZMGrQ==|03 |storage2|[DflObject, TimeSeries, NumericTimeSeries]|
PCQcM/abQFKtP0MyXywZrA==|03 |main |[DflObject, TimeSeries, NumericTimeSeries]|
vXYNGpHLQluai9kyVW1oHA==|03 |storage2|[DflObject, TimeSeries, NumericTimeSeries]|
+------------------------+-----+--------+------------------------------------------+

+------------------------+-----+-------+-----------------------------------------+
$binary |$type|Storage|_t |
+------------------------+-----+-------+-----------------------------------------+
cEAISZugnU2/YfqyEP8dMg==|03 |main |[DflObject, TimeSeries, StringTimeSeries]|
+------------------------+-----+-------+-----------------------------------------+

### Events-> eventsTimestampedEntry : Metadata device IDs read

In [0]:
if (metadataCollection == 'events' and telemetryCollection == 'eventsTimestampedEntry') :
  eventsDF1 = spark.read.json(metadataPath)
  eventsDF2 = eventsDF1.select("_id.*","Storage","_t")
  eventsDF = eventsDF2.select("$binary","Storage",eventsDF2["_t"].getItem(2).alias("type"))
  storageTimestampedEvents = []  
  for i in range(len(storages[0])):
    eventsTimestampedNames = {}
    storageName = storages[0][i]["Name"]
    dfTimestamped = eventsDF.filter((col('Storage') == storageName) & (col('type') == "TimestampedEventSeries")).select(col("$binary").alias("_id"))
    eventsTimestampedIDs = dfTimestamped.select("_id").rdd.flatMap(lambda x: x).collect()
    eventsTimestampedName = "events_timestamped" + storageName + "IDs"
    eventsTimestampedNames[eventsTimestampedName] = eventsTimestampedIDs  
    storageTimestampedEvents.append(eventsTimestampedNames)

### Events-> eventsIntervalEntry : Metadata device IDs read

In [0]:
if (metadataCollection == 'events' and telemetryCollection == 'eventsIntervalEntry') :
  eventsDF1 = spark.read.json(metadataPath)
  eventsDF2 = eventsDF1.select("_id.*","Storage","_t")
  eventsDF = eventsDF2.select("$binary","Storage",eventsDF2["_t"].getItem(2).alias("type"))
  storageIntervalEvents = []
  for i in range(len(storages[0])):
    eventsIntervalNames = {}
    storageName = storages[0][i]["Name"]
    dfInterval = eventsDF.filter((col('Storage') == storageName) & (col('type') == "IntervalEventSeries")).select(col("$binary").alias("_id"))
    eventsIntervalIDs = dfInterval.select("_id").rdd.flatMap(lambda x: x).collect()
    eventsIntervalName = "events_interval" + storageName + "IDs"
    eventsIntervalNames[eventsIntervalName] = eventsIntervalIDs  
    storageIntervalEvents.append(eventsIntervalNames)

### Function to convert CSHARP_LEGACY UUID to Standard Format

In [0]:
@udf("String")
def parseId(id_base64):
  id_hex = base64.b64decode(id_base64).hex()
  a = id_hex[6:8] + id_hex[4:6] + id_hex[2:4] + id_hex[0:2]
  b = id_hex[10:12] + id_hex[8:10]
  c = id_hex[14:16] + id_hex[12:14]
  d = id_hex[16:32]
  id_hex = a + b + c + d
  id = id_hex[0:8] + '-' + id_hex[8:12] + '-' + id_hex[12:16] + '-' + id_hex[16:20] + '-' + id_hex[20:32]
  return id

### UDF to have only single space after fields end in line protocol

In [0]:
@udf("String")
def removeComma(lineProtocol):
  if lineProtocol[-1] == ',':
    temp = list(lineProtocol)
    temp[-1] = ''
    new_str = "".join(temp)
    return new_str
  else :
    return lineProtocol

### Function to convert pyspark dataframe to Influx Line Protocol

In [0]:
def toLineProtocolNew(seriesType, DF : DataFrame, tag_columns, field_columns, timestamp) -> DataFrame:
  #lineProtocolDF = DF.withColumn("storage",lit(storage)).select("*")
  lineProtocolDF = DF.select("*")
  for i in range(len(tag_columns)):
    lineProtocolDF = lineProtocolDF.withColumn("tag_" + tag_columns[i],concat(lit(","),lit(tag_columns[i]),lit("="),lineProtocolDF[tag_columns[i]])).select("*").drop(tag_columns[i])
  lineProtocolDF = lineProtocolDF.withColumn("tagsEnd",lit(" ")).select("*")
  fieldCount = len(field_columns)
  for i in range(len(field_columns)):
    columnName = field_columns[i]
    print("TEST PRINTING CONDITIONAL VARS")
    print(columnName)
    print(seriesType)
    if ((columnName == 'data' and seriesType == 'NumericTimeSeries') or (columnName == 'end_time')):
      lineProtocolDF = lineProtocolDF.withColumn("field_" + field_columns[i],concat(lit(field_columns[i]),lit("="),lineProtocolDF[field_columns[i]])).select("*")
    else :
      lineProtocolDF = lineProtocolDF.withColumn("field_" + field_columns[i],concat(lit(field_columns[i]),lit("="),lit('\"'),lineProtocolDF[field_columns[i]],lit('\"'))).select("*")
    fieldCount = fieldCount - 1   
    if fieldCount > 0:
      lineProtocolDF = lineProtocolDF.withColumn("field_" + field_columns[i],concat(lineProtocolDF["field_" + field_columns[i]],lit(","))).select("*")
  lineProtocolDF = lineProtocolDF.withColumn("fieldsEnd",lit(" ")).select("*")
  lineProtocolDF = lineProtocolDF.withColumn("lineProtocol",lineProtocolDF['storage']).select("*")
  for i in range(len(tag_columns)):
    lineProtocolDF = lineProtocolDF.withColumn("lineProtocol",concat_ws('',lineProtocolDF["lineProtocol"],lineProtocolDF["tag_" + tag_columns[i]])).select("*")
  lineProtocolDF = lineProtocolDF.withColumn("lineProtocol",concat(lineProtocolDF["lineProtocol"],lineProtocolDF['tagsEnd'])).select("*")
  for i in range(len(field_columns)):
    lineProtocolDF = lineProtocolDF.withColumn("lineProtocol",concat_ws('',lineProtocolDF["lineProtocol"],lineProtocolDF["field_" + field_columns[i]])).select("*")
  lineProtocolDF.printSchema()
  lineProtocolDF = lineProtocolDF.select(removeComma("lineProtocol").alias("lineProtocol"),"fieldsEnd",timestamp,"storage")
  lineProtocolDF = lineProtocolDF.withColumn("lineProtocol",concat(lineProtocolDF["lineProtocol"],lineProtocolDF['fieldsEnd'])).select("lineProtocol",timestamp,"storage")
  lineProtocolDF.printSchema()
  lineProtocolDF = lineProtocolDF.withColumn("lineProtocol",concat(lineProtocolDF["lineProtocol"],lineProtocolDF[timestamp])).select("lineProtocol","storage")
  return lineProtocolDF

### seriesEntry : Fetch telemetry, convert into line protocol and write into ADLS

In [0]:
if telemetryCollection == 'seriesEntry':
    entry = spark.read.json(telemetryEntryPath)
    #entry.printSchema()
    #entry.show(truncate=False)
    #entry.createOrReplaceTempView("entry_raw_int_table")
    if 'array' not in entry.select('a').dtypes[0][1]:
        badEntries = None
        try:
          badEntries = entry.select(col('_id'), func.substring_index(col('a'), '_v":', -1).alias('a'), col('s'), col('t'),
                                      col('v')).filter(entry.a.contains('_t'))
          badEntries = badEntries.select(col('_id'), func.expr("substring(a,0,length(a)-1)").alias('a'), col('s'), col('t'),
                                           col('v'))
          goodEntries = entry.select('*').filter(~entry.a.contains('_t') | entry.a.isNull())
          refinedEntries = goodEntries.union(badEntries)
          att_schema = ArrayType(StructType([StructField("Name", StringType()), StructField("SemanticRef", StringType()),
                                               StructField("Value", StringType())]))
          refinedEntries = refinedEntries.withColumn("a", func.from_json(col('a'), att_schema)).createOrReplaceTempView("entry_raw_seriesEntry_table")
        except:
            refinedEntries = badEntries
            att_schema = ArrayType(StructType([StructField("Name", StringType()), StructField("SemanticRef", StringType()),
                                             StructField("Value", StringType())]))
            refinedEntries = refinedEntries.withColumn("a", func.from_json(col('a'), att_schema)).createOrReplaceTempView("entry_raw_seriesEntry_table")
    else:
        entry.createOrReplaceTempView("entry_raw_seriesEntry_table")

  #####WORKING CODE FOR INTSERIES#####################################
    noAttrFlag = 0
    try:
      spark.sql("select * from (select tab1.`_id`,tab1.s.`$binary` as s, tab1.v, tab1.t, tab1.a, tab2.storage,size(tab1.a) as a_size from entry_raw_seriesEntry_table tab1  join series_temp_table_numeric_series tab2 on trim(tab1.s.`$binary`) = trim(tab2.`$binary`)) ").createOrReplaceTempView("entry_raw_seriesEntry_table_with_a_size")
      seriesIntEntryDF1 = spark.sql("select `_id`, s, v, t, a, storage from entry_raw_seriesEntry_table_with_a_size where a_size >= 1")
      seriesIntEmptyAEntryDF1 = spark.sql("select `_id`, s, v, t, a, storage from entry_raw_seriesEntry_table_with_a_size where a_size = 0 or a is null")
      #seriesIntEntryDF1.show(truncate=False)
      #seriesIntEmptyAEntryDF1.show(truncate=False)
      #print(seriesIntEmptyAEntryDF1.count())
    except:
      noAttrFlag = 1
      seriesIntEmptyAEntryDF1 = spark.sql("select s, v, t storage from entry_raw_seriesEntry_table")
      
    

    if (noAttrFlag != 1) and (len(seriesIntEntryDF1.head(1)) != 0):
        seriesIntEntryDF2 = seriesIntEntryDF1.select(col("s"), col("v"), col("t"),col("storage"), func.explode(col("a")))
        seriesIntEntry = seriesIntEntryDF2.select(col("s"), col("v"), col("t.*"),col("storage"), col("col.*"))
        seriesIntEntryNullsRemoved = seriesIntEntry.na.fill("")
        seriesIntEntryNullsRemovedTrans1 = seriesIntEntryNullsRemoved.select("v", "SemanticRef", "$date", "Name", "s", "Value", "storage").groupBy("s","$date","Name","v","storage").pivot("Name").agg(func.first('SemanticRef').alias('SemanticRef'), func.first('Value').alias('Value'))
        #seriesIntEntryNullsRemovedTrans1.show(truncate=False)
        seriesIntEntryNullsRemovedTrans2 = seriesIntEntryNullsRemovedTrans1.select(parseId("s").alias("series_id"), col("v").alias("data"), col("$date").alias("_timestamp"), "*").drop("s", "v", "$date")
        #seriesIntEntryNullsRemovedTrans2.show(truncate=False)

        oldColumns = seriesIntEntryNullsRemovedTrans2.schema.names
        mapping = {}
        flattenedColumns = []
        for i in range(len(oldColumns)):
          if (oldColumns[i].endswith("SemanticRef")) or (oldColumns[i].endswith("Value")):
            new_col = "attr_" + oldColumns[i]
            sstring = "_Value"
            if new_col.endswith(sstring):
              new_col = new_col.replace(sstring, '')
            sr_sub_str = "SemanticRef"
            if new_col.endswith(sr_sub_str):
              new_col = new_col.replace(sr_sub_str, 'semanticref')
            mapping[oldColumns[i]] = new_col
            flattenedColumns.append(new_col)
          else:
            mapping[oldColumns[i]] = oldColumns[i]
        seriesIntEntryNullsRemovedTrans3 = seriesIntEntryNullsRemovedTrans2.select([col(c).alias(mapping.get(c, c)) for c in seriesIntEntryNullsRemovedTrans2.columns])
        field_cols = fields.split(",")
        tag_cols = tags.split(",")
        if 'attr_' in field_cols:
          field_cols.remove('attr_')
          field_cols.extend(flattenedColumns)

        if 'attr_' in tag_cols:
          tag_cols.remove('attr_')
          tag_cols.extend(flattenedColumns)
        print(mapping)
        print(flattenedColumns)

        lineProtocolIntDF = toLineProtocolNew("NumericTimeSeries", seriesIntEntryNullsRemovedTrans3, tag_cols, field_cols,"_timestamp")
        #lineProtocolIntDF.show(truncate=False)
        lineProtocolIntDF.write.format("text").option("header", "false").partitionBy("storage").mode("append").save(lineProtocolPath)
    
    #---------------Write Empty Attribute Int entries to disk ##
    if len(seriesIntEmptyAEntryDF1.head(1)) != 0:
      empty_Attr_tag_cols = tags.split(",")
      if 'attr_' in empty_Attr_tag_cols:
        empty_Attr_tag_cols.remove('attr_')
 
      empty_Attr_field_cols = fields.split(",")
      if 'attr_' in empty_Attr_field_cols:
        empty_Attr_field_cols.remove('attr_')
      seriesIntEmptyAEntryDF2 = seriesIntEmptyAEntryDF1.select(parseId("s").alias("series_id"), col("v").alias("data"), col("t.$date").alias("_timestamp"), "*").drop("s", "v","t")
      seriesIntEmptyAEntryDF2.printSchema()
      lineProtocolIntEmptyADF = toLineProtocolNew("NumericTimeSeries", seriesIntEmptyAEntryDF2,empty_Attr_tag_cols, empty_Attr_field_cols, "_timestamp")
      #lineProtocolIntEmptyADF.show(truncate=False)
      lineProtocolIntEmptyADF.write.format("text").option("header", "false").partitionBy("storage").mode("append").save(lineProtocolPath)
    
    #########String Series Data Processing############################
    
    noAttrFlag = 0
    try:
      spark.sql("select * from (select tab1.`_id`,tab1.s.`$binary` as s, tab1.v, tab1.t, tab1.a, tab2.storage,size(tab1.a) as a_size from entry_raw_seriesEntry_table tab1  join series_temp_table_string_series tab2 on trim(tab1.s.`$binary`) = trim(tab2.`$binary`)) ").createOrReplaceTempView("entry_raw_seriesEntry_string_table_with_a_size")
      seriesStrEntryDF1 = spark.sql("select `_id`, s, v, t, a, storage from entry_raw_seriesEntry_string_table_with_a_size where a_size >= 1")
      seriesStrEmptyAEntryDF1 = spark.sql("select `_id`, s, v, t, a, storage from entry_raw_seriesEntry_string_table_with_a_size where a_size = 0 or a is null")
      seriesStrEntryDF1.show(truncate=False)
      seriesStrEmptyAEntryDF1.show(truncate=False)
      print(seriesStrEmptyAEntryDF1.count())
    except:
      noAttrFlag = 1
      seriesStrEmptyAEntryDF1 = spark.sql("select s, v, t storage from entry_raw_seriesEntry_table")
        
    #print("COUNTING STRING ENTRIES")
    #print(seriesStrEntryDF1.count())
    
    if (noAttrFlag != 1) and (len(seriesStrEntryDF1.head(1)) != 0):
      tag_cols = tags.split(",")
      field_cols = fields.split(",")
      seriesStrEntryDF2 = seriesStrEntryDF1.select(col("s"), col("v"), col("t"),col("storage"), func.explode(col("a")))
      seriesStrEntry = seriesStrEntryDF2.select(col("s"), col("v"), col("t.*"),col("storage"), col("col.*"))
      seriesStrEntryNullsRemoved = seriesStrEntry.na.fill("")
      seriesStrEntryNullsRemovedTrans1 = seriesStrEntryNullsRemoved.select("v", "SemanticRef", "$date", "Name", "s", "Value", "storage").groupBy("s","$date","Name","v","storage").pivot("Name").agg(func.first('SemanticRef').alias('SemanticRef'), func.first('Value').alias('Value'))
      #seriesStrEntryNullsRemovedTrans1.show(truncate=False)
      seriesStrEntryNullsRemovedTrans2 = seriesStrEntryNullsRemovedTrans1.select(parseId("s").alias("series_id"), col("v").alias("data"), col("$date").alias("_timestamp"), "*").drop("s", "v", "$date")
      #seriesStrEntryNullsRemovedTrans2.show(truncate=False)

      oldColumns = seriesStrEntryNullsRemovedTrans2.schema.names
      mapping = {}
      flattenedColumns = []
      for i in range(len(oldColumns)):
        if (oldColumns[i].endswith("SemanticRef")) or (oldColumns[i].endswith("Value")):
          new_col = "attr_" + oldColumns[i]
          sstring = "_Value"
          if new_col.endswith(sstring):
            new_col = new_col.replace(sstring, '')
          sr_sub_str = "SemanticRef"
          if new_col.endswith(sr_sub_str):
            new_col = new_col.replace(sr_sub_str, 'semanticref')
          mapping[oldColumns[i]] = new_col
          flattenedColumns.append(new_col)
        else:
          mapping[oldColumns[i]] = oldColumns[i]
      seriesStrEntryNullsRemovedTrans3 = seriesStrEntryNullsRemovedTrans2.select(
        [col(c).alias(mapping.get(c, c)) for c in seriesStrEntryNullsRemovedTrans2.columns])
      if 'attr_' in field_cols:
        field_cols.remove('attr_')
        field_cols.extend(flattenedColumns)
 
      if 'attr_' in tag_cols:
        tag_cols.remove('attr_')
        tag_cols.extend(flattenedColumns)
      
      #print("seriesStrEntryNullsRemovedTrans3 SCHEMA AND DATA")
      #seriesStrEntryNullsRemovedTrans3.printSchema()
      #seriesStrEntryNullsRemovedTrans3.show(truncate=False)
      #print(tag_cols)
      #print(field_cols)
      lineProtocolStrDF = toLineProtocolNew("StringTimeSeries", seriesStrEntryNullsRemovedTrans3, tag_cols, field_cols,"_timestamp")
      #lineProtocolStrDF.show(truncate=False)
      lineProtocolStrDF.write.format("text").option("header", "false").partitionBy("storage").mode("append").save(lineProtocolPath)
    
    #print("COUNTING STRING EMPTY_A ENTRIES")
    #print(seriesStrEntryDF1.count())
    
    if len(seriesStrEmptyAEntryDF1.head(1)) != 0:
      empty_Attr_tag_cols = tags.split(",")
      if 'attr_' in empty_Attr_tag_cols:
        empty_Attr_tag_cols.remove('attr_')
 
      empty_Attr_field_cols = fields.split(",")
      if 'attr_' in empty_Attr_field_cols:
        empty_Attr_field_cols.remove('attr_')
      seriesStrEmptyAEntryDF2 = seriesStrEmptyAEntryDF1.select(parseId("s").alias("series_id"), col("v").alias("data"), col("t.$date").alias("_timestamp"), "*").drop("s", "v","t")
      #seriesIntEmptyAEntryDF2.printSchema()
      lineProtocolStrEmptyADF = toLineProtocolNew("StringTimeSeries", seriesStrEmptyAEntryDF2,empty_Attr_tag_cols, empty_Attr_field_cols, "_timestamp")
      #lineProtocolStrEmptyADF.show(truncate=False)
      lineProtocolStrEmptyADF.write.format("text").option("header", "false").partitionBy("storage").mode("append").save(lineProtocolPath)
    
    



root
-- _id: struct (nullable = true)
 |-- $oid: string (nullable = true)
-- a: string (nullable = true)
-- s: struct (nullable = true)
 |-- $binary: string (nullable = true)
 |-- $type: string (nullable = true)
-- t: struct (nullable = true)
 |-- $date: long (nullable = true)
-- v: string (nullable = true)

+--------------------------+-------------------------------------------------+------------------------------+---------------+-------------------+
_id |a |s |t |v |
+--------------------------+-------------------------------------------------+------------------------------+---------------+-------------------+
{600f477ff887230f242daa15}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577838794000}|0.5710711036366942 |
{600f477ff887230f242d9db0}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577842394000}|0.7453323650526735 |
{600f477ff887230f242da531}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577845994000}|0.2229338653124 |
{600f477ff887230f242dbb4c}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577849594000}|1.0333072433921113 |
{600f477ff887230f242daf0f}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577853194000}|1.2350611920234147 |
{600f477ff887230f242ddb21}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577856794000}|0.3950647358061026 |
{600f477ff887230f242dd654}|[{"Name":"Quality","SemanticRef":"","Value":"0"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577860394000}|1.881217407828511 |
{600f477ff887230f242dc7a4}|[{"Name":"Quality","SemanticRef":"","Value":"0"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577863994000}|1.2651265233768272 |
{600f477ff887230f242dc047}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577867594000}|0.44654002436781814|
{600f477ff887230f242dd8c0}|[{"Name":"Quality","SemanticRef":"","Value":"0"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577871194000}|1.3356097954853758 |
{600f477ff887230f242ddffb}|[{"Name":"Quality","SemanticRef":"","Value":"0"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577874794000}|0.6844753254650504 |
{600f477ff887230f242db7ef}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577878394000}|0.9313650730575456 |
{600f477ff887230f242dc2bf}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577881994000}|0.9046706040688133 |
{600f477ff887230f242dbdcf}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577885594000}|1.536733501347841 |
{600f477ff887230f242de26e}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577889194000}|0.7080673292824453 |
{600f477ff887230f242db1db}|[{"Name":"Quality","SemanticRef":"","Value":"0"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577892794000}|0.988020479369041 |
{600f477ff887230f242db746}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577896394000}|1.7410116818565677 |
{600f477ff887230f242da13e}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577899994000}|1.217703106444505 |
{600f477ff887230f242dcc6b}|[{"Name":"Quality","SemanticRef":"","Value":"1"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577903594000}|0.541182862353266 |
{600f477ff887230f242da7a4}|[{"Name":"Quality","SemanticRef":"","Value":"0"}]|{EgCYT+PgQpG5BWfqPcDOFA==, 03}|{1577907194000}|0.8602830389225569 |
+--------------------------+-------------------------------------------------+------------------------------+---------------+-------------------+
only showing top 20 rows

{'series_id': 'series_id', 'data': 'data', '_timestamp': '_timestamp', 'Name': 'Name', 'storage': 'storage', 'InsertedTime_SemanticRef': 'attr_InsertedTime_semanticref', 'InsertedTime_Value': 'attr_InsertedTime', 'Quality_SemanticRef': 'attr_Quality_semanticref', 'Quality_Value': 'attr_Quality', 'Site_SemanticRef':

### eventsIntervalEntry : Fetch telemetry, convert into line protocol and write into ADLS

In [0]:
if telemetryCollection == 'eventsIntervalEntry':
  entry = spark.read.json(telemetryEntryPath)
  for storage in range(len(storages[0])):
    #-
    storageName = storages[0][storage]["Name"]
    eventsIntervalName = "events_interval" + storageName + "IDs"
    intervalIds = storageIntervalEvents[storage].get(eventsIntervalName) 
    noAttrFlag = 0
    
    try :
      eventsIntervalEntryDF1 = entry.filter(col('s.$binary').isin(intervalIds)).select(col("s.$binary").alias("s"),col("_id.$binary").alias("_id"),col("d"),col("st.$date").alias("st"),col("et.$date").alias("et"),col("es"),col("a"))
      eventsIntervalEntryDF2 = eventsIntervalEntryDF1.withColumn("a_size", func.size(func.col("a")))
      # To filter records with attributes array
      eventsIntervalEntryTF1 = eventsIntervalEntryDF2.filter(func.col("a_size") >= 1).drop("a_size")
      # To filter records with an empty attributes array
      eventsIntervalEmptyAEntryTF1 = eventsIntervalEntryDF2.filter((func.col("a_size") == 0) | (func.col("a").isNull())).drop("a_size")
    except Exception as e :
      try :
        eventsIntervalEntryDF1 = entry.filter(col('s.$binary').isin(intervalIds)).select(col("s.$binary").alias("s"),col("_id.$binary").alias("_id"),col("d"),col("st.$date").alias("st"),col("et.$date").alias("et"),col("a"))
        eventsIntervalEntryDF2 = eventsIntervalEntryDF1.withColumn("a_size", func.size(func.col("a")))
        # To filter records with attributes array
        eventsIntervalEntryTF1 = eventsIntervalEntryDF2.filter(func.col("a_size") >= 1).drop("a_size")
        # To filter records with an empty attributes array
        eventsIntervalEmptyAEntryTF1 = eventsIntervalEntryDF2.filter((func.col("a_size") == 0) | (func.col("a").isNull())).drop("a_size")

      except Exception as e :
        noAttrFlag = 1
        try :
          eventsIntervalEmptyAEntryTF1 = entry.filter(col('s.$binary').isin(intervalIds)).select(col("s.$binary").alias("s"),col("_id.$binary").alias("_id"),col("d"),col("st.$date").alias("st"),col("et.$date").alias("et"),col("es"))
        except :
          eventsIntervalEmptyAEntryTF1 = entry.filter(col('s.$binary').isin(intervalIds)).select(col("s.$binary").alias("s"),col("_id.$binary").alias("_id"),col("d"),col("st.$date").alias("st"),col("et.$date").alias("et"))
    #---
    if (noAttrFlag != 1) and (len(eventsIntervalEntryTF1.head(1)) != 0): 
      try :
        eventsIntervalEntryTF2 = eventsIntervalEntryTF1.select(col("s"),col("_id"),col("d"),col("st"),col("et"),col("es"),func.explode(col("a")))
        eventsIntervalEntry = eventsIntervalEntryTF2.select(col("s"),col("es"),col("_id"),col("d"),col("st"),col("et"),col("col.*"))
        eventsIntervalEntryDFTF = eventsIntervalEntry.na.fill("",["Name"]).na.fill("",["SemanticRef"]).na.fill("",["Value"])
        eventsIntervalEntryDFTF1 = eventsIntervalEntryDFTF.select("s","es","_id","d","st","et","Name","SemanticRef","Value").groupBy("s","es","_id","st","et","Name","d").pivot("Name").agg(func.first('SemanticRef').alias('SemanticRef'),func.first('Value').alias('Value'))
        eventsIntervalEntryDFTF2= eventsIntervalEntryDFTF1.select(parseId("s").alias("series_id"),col("es").alias("source_id"),parseId("_id").alias("instance_id"),col("d").alias("data"),col("st").alias("_timestamp"),col("et").alias("end_time"),"*").drop("es","s","_id","d","st","et")

        oldColumns = eventsIntervalEntryDFTF2.schema.names
        mapping = {}
        flattenedColumns = []
        for i in range(len(oldColumns)):
          if (oldColumns[i].endswith("SemanticRef")) or (oldColumns[i].endswith("Value")):
            new_col = "attr_" + oldColumns[i]
            sstring = "_Value"
            if new_col.endswith(sstring): 
              new_col = new_col.replace(sstring, '') 
            sr_sub_str = "SemanticRef"
            if new_col.endswith(sr_sub_str): 
              new_col = new_col.replace(sr_sub_str, 'semanticref') 
            mapping[oldColumns[i]] = new_col
            flattenedColumns.append(new_col)
          else :
            mapping[oldColumns[i]]=oldColumns[i]
        eventsIntervalEntryDFTF3 = eventsIntervalEntryDFTF2.select([col(c).alias(mapping.get(c, c)) for c in eventsIntervalEntryDFTF2.columns])
        
        field_cols = fields.split(",")
        tag_cols = tags.split(",")
     

        if 'attr_' in field_cols :
          field_cols.remove('attr_')
          field_cols.extend(flattenedColumns)

        if 'attr_' in tag_cols :
          tag_cols.remove('attr_')
          tag_cols.extend(flattenedColumns)
        lineProtocolIntervalDF = toLineProtocol(storageName, "IntervalEventSeries", eventsIntervalEntryDFTF3, tag_cols, field_cols, "_timestamp")
        storagePath = lineProtocolPath + "/" + storageName
        lineProtocolIntervalDF.coalesce(12).write.format("text").option("header", "false").mode("append").save(storagePath)
      except Exception as e:
        eventsIntervalEntryTF2 = eventsIntervalEntryTF1.select(col("s"),col("_id"),col("d"),col("st"),col("et"),func.explode(col("a")))
        eventsIntervalEntry = eventsIntervalEntryTF2.select(col("s"),col("_id"),col("d"),col("st"),col("et"),col("col.*"))
        eventsIntervalEntryDFTF = eventsIntervalEntry.na.fill("",["Name"]).na.fill("",["SemanticRef"]).na.fill("",["Value"])
        eventsIntervalEntryDFTF1 = eventsIntervalEntryDFTF.select("s","_id","d","st","et","Name","SemanticRef","Value").groupBy("s","_id","st","et","Name","d").pivot("Name").agg(func.first('SemanticRef').alias('SemanticRef'),func.first('Value').alias('Value'))
        eventsIntervalEntryDFTF2= eventsIntervalEntryDFTF1.select(parseId("s").alias("series_id"),parseId("_id").alias("instance_id"),col("d").alias("data"),col("st").alias("_timestamp"),col("et").alias("end_time"),"*").drop("s","_id","d","st","et")

        oldColumns = eventsIntervalEntryDFTF2.schema.names
        mapping = {}
        flattenedColumns = []
        for i in range(len(oldColumns)):
          if (oldColumns[i].endswith("SemanticRef")) or (oldColumns[i].endswith("Value")):
            new_col = "attr_" + oldColumns[i]
            sstring = "_Value"
            if new_col.endswith(sstring): 
              new_col = new_col.replace(sstring, '') 
            sr_sub_str = "SemanticRef"
            if new_col.endswith(sr_sub_str): 
              new_col = new_col.replace(sr_sub_str, 'semanticref') 
            mapping[oldColumns[i]] = new_col
            flattenedColumns.append(new_col)
          else :
            mapping[oldColumns[i]]=oldColumns[i]
        eventsIntervalEntryDFTF3 = eventsIntervalEntryDFTF2.select([col(c).alias(mapping.get(c, c)) for c in eventsIntervalEntryDFTF2.columns])
        field_cols = fields.split(",")
        tag_cols = tags.split(",")
        tag_cols.remove('source_id')
        

        if 'attr_' in field_cols :
          field_cols.remove('attr_')
          field_cols.extend(flattenedColumns)

        if 'attr_' in tag_cols :
          tag_cols.remove('attr_')
          tag_cols.extend(flattenedColumns)
     
        lineProtocolIntervalDF = toLineProtocol(storageName, "IntervalEventSeries", eventsIntervalEntryDFTF3, tag_cols, field_cols, "_timestamp")
        storagePath = lineProtocolPath + "/" + storageName
        lineProtocolIntervalDF.coalesce(12).write.format("text").option("header", "false").mode("append").save(storagePath)
        
    if len(eventsIntervalEmptyAEntryTF1.head(1)) != 0: 
      empty_Attr_tag_cols = tags.split(",")
    
      if 'attr_' in empty_Attr_tag_cols :
        empty_Attr_tag_cols.remove('attr_')

      empty_Attr_field_cols = fields.split(",")
      if 'attr_' in empty_Attr_field_cols :   
        empty_Attr_field_cols.remove('attr_')
      try :
        eventsIntervalEmptyAEntryTF2 = eventsIntervalEmptyAEntryTF1.select(parseId("s").alias("series_id"),col("es").alias("source_id"),parseId("_id").alias("instance_id"),col("d").alias("data"),col("st").alias("_timestamp"),col("et").alias("end_time")).drop("es","s","_id","d","st","et")
      except Exception as e :
        empty_Attr_tag_cols.remove('source_id')
        eventsIntervalEmptyAEntryTF2 = eventsIntervalEmptyAEntryTF1.select(parseId("s").alias("series_id"),parseId("_id").alias("instance_id"),col("d").alias("data"),col("st").alias("_timestamp"),col("et").alias("end_time")).drop("s","_id","d","st","et")
      lineProtocolEmptyIntervalDF = toLineProtocol(storageName, "IntervalEventSeries", eventsIntervalEmptyAEntryTF2, empty_Attr_tag_cols, empty_Attr_field_cols, "_timestamp")
      storagePath = lineProtocolPath + "/" + storageName
      lineProtocolEmptyIntervalDF.coalesce(12).write.format("text").option("header", "false").mode("append").save(storagePath)

### eventsTimestampedEntry : Fetch telemetry, convert into line protocol and write into ADLS

In [0]:
if telemetryCollection == 'eventsTimestampedEntry':
  entry = spark.read.json(telemetryEntryPath)
  for storage in range(len(storages[0])):
    storageName = storages[0][storage]["Name"]
    eventsTimestampedName = "events_timestamped" + storageName + "IDs"
    timestampedIds = storageTimestampedEvents[storage].get(eventsTimestampedName)
    noAttrFlag = 0
   
    #---
    try :
      eventsTimestampedEntryDF1 = entry.filter(col('s.$binary').isin(timestampedIds)).select(col("s.$binary").alias("s"),col("_id.$binary").alias("_id"),col("d"),col("t.$date").alias("t"),col("es"),col("a"))
      eventsTimestampedEntryDF2 = eventsTimestampedEntryDF1.withColumn("a_size", func.size(func.col("a")))
      # To filter records with attributes array
      eventsTimestampedEntryTF1 = eventsTimestampedEntryDF2.filter(func.col("a_size") >= 1).drop("a_size")
      # To filter records with an empty attributes array
      eventsTimestampedEmptyAEntryTF1 = eventsTimestampedEntryDF2.filter((func.col("a_size") == 0)| (func.col("a").isNull())).drop("a_size")
    except Exception as e:
      try :
        eventsTimestampedEntryDF1 = entry.filter(col('s.$binary').isin(timestampedIds)).select(col("s.$binary").alias("s"),col("_id.$binary").alias("_id"),col("d"),col("t.$date").alias("t"),col("a"))
        eventsTimestampedEntryDF2 = eventsTimestampedEntryDF1.withColumn("a_size", func.size(func.col("a")))
        # To filter records with attributes array
        eventsTimestampedEntryTF1 = eventsTimestampedEntryDF2.filter(func.col("a_size") >= 1).drop("a_size")
        # To filter records with an empty attributes array
        eventsTimestampedEmptyAEntryTF1 = eventsTimestampedEntryDF2.filter((func.col("a_size") == 0)| (func.col("a").isNull())).drop("a_size")
      except :
        noAttrFlag = 1
        try :
          eventsTimestampedEmptyAEntryTF1 = entry.filter(col('s.$binary').isin(timestampedIds)).select(col("s.$binary").alias("s"),col("_id.$binary").alias("_id"),col("d"),col("t.$date").alias("t"),col("es"))
        except :
          eventsTimestampedEmptyAEntryTF1 = entry.filter(col('s.$binary').isin(timestampedIds)).select(col("s.$binary").alias("s"),col("_id.$binary").alias("_id"),col("d"),col("t.$date").alias("t"))
    #--
    if (noAttrFlag !=1) and len(eventsTimestampedEntryTF1.head(1)) != 0: 
      try :
        eventsTimestampedEntryTF2 = eventsTimestampedEntryTF1.select(col("s"),col("_id"),col("d"),col("t"),col("es"),func.explode(col("a")))
        eventsTimestampedEntry = eventsTimestampedEntryTF2.select(col("s"),col("es"),col("_id"),col("d"),col("t"),col("col.*"))
        eventsTimestampedEntryDFTF = eventsTimestampedEntry.na.fill("",["Name"]).na.fill("",["SemanticRef"]).na.fill("",["Value"])
        eventsTimestampedEntryDFTF1 = eventsTimestampedEntryDFTF.select("s","es","_id","d","t","Name","SemanticRef","Value").groupBy("s","es","_id","t","Name","d").pivot("Name").agg(func.first('SemanticRef').alias('SemanticRef'),func.first('Value').alias('Value'))
        eventsTimestampedEntryDFTF2= eventsTimestampedEntryDFTF1.select(parseId("s").alias("series_id"),col("es").alias("source_id"),parseId("_id").alias("instance_id"),col("d").alias("data"),col("t").alias("_timestamp"),"*").drop("es","s","_id","d","t")

        oldColumns = eventsTimestampedEntryDFTF2.schema.names
        mapping = {}
        flattenedColumns = []
        for i in range(len(oldColumns)):
          if (oldColumns[i].endswith("SemanticRef")) or (oldColumns[i].endswith("Value")):
            new_col = "attr_" + oldColumns[i]
            sstring = "_Value"
            if new_col.endswith(sstring): 
              new_col = new_col.replace(sstring, '') 
            sr_sub_str = "SemanticRef"
            if new_col.endswith(sr_sub_str): 
              new_col = new_col.replace(sr_sub_str, 'semanticref') 
            mapping[oldColumns[i]] = new_col
            flattenedColumns.append(new_col)
          else :
            mapping[oldColumns[i]]=oldColumns[i]
        eventsTimestampedEntryDFTF3 = eventsTimestampedEntryDFTF2.select([col(c).alias(mapping.get(c, c)) for c in eventsTimestampedEntryDFTF2.columns])
        field_cols = fields.split(",")
        tag_cols = tags.split(",")

        if 'attr_' in field_cols :
          field_cols.remove('attr_')
          field_cols.extend(flattenedColumns)

        if 'attr_' in tag_cols :
          tag_cols.remove('attr_')
          tag_cols.extend(flattenedColumns)
        lineProtocolTimestampedDF = toLineProtocol(storageName, "TimestampedEventSeries", eventsTimestampedEntryDFTF3, tag_cols, field_cols, "_timestamp")
        storagePath = lineProtocolPath + "/" + storageName
        lineProtocolTimestampedDF.coalesce(12).write.format("text").option("header", "false").mode("append").save(storagePath)
      except Exception as e :
        eventsTimestampedEntryTF2 = eventsTimestampedEntryTF1.select(col("s"),col("_id"),col("d"),col("t"),func.explode(col("a")))
        eventsTimestampedEntry = eventsTimestampedEntryTF2.select(col("s"),col("_id"),col("d"),col("t"),col("col.*"))
        eventsTimestampedEntryDFTF = eventsTimestampedEntry.na.fill("",["Name"]).na.fill("",["SemanticRef"]).na.fill("",["Value"])
        eventsTimestampedEntryDFTF1 = eventsTimestampedEntryDFTF.select("s","_id","d","t","Name","SemanticRef","Value").groupBy("s","_id","t","Name","d").pivot("Name").agg(func.first('SemanticRef').alias('SemanticRef'),func.first('Value').alias('Value'))
        eventsTimestampedEntryDFTF2= eventsTimestampedEntryDFTF1.select(parseId("s").alias("series_id"),parseId("_id").alias("instance_id"),col("d").alias("data"),col("t").alias("_timestamp"),"*").drop("s","_id","d","t")

        oldColumns = eventsTimestampedEntryDFTF2.schema.names
        mapping = {}
        flattenedColumns = []
        for i in range(len(oldColumns)):
          if (oldColumns[i].endswith("SemanticRef")) or (oldColumns[i].endswith("Value")):
            new_col = "attr_" + oldColumns[i]
            sstring = "_Value"
            if new_col.endswith(sstring): 
              new_col = new_col.replace(sstring, '') 
            sr_sub_str = "SemanticRef"
            if new_col.endswith(sr_sub_str): 
              new_col = new_col.replace(sr_sub_str, 'semanticref') 
            mapping[oldColumns[i]] = new_col
            flattenedColumns.append(new_col)
          else :
            mapping[oldColumns[i]]=oldColumns[i]
        eventsTimestampedEntryDFTF3 = eventsTimestampedEntryDFTF2.select([col(c).alias(mapping.get(c, c)) for c in eventsTimestampedEntryDFTF2.columns])
        field_cols = fields.split(",")
        tag_cols = tags.split(",")
        tag_cols.remove('source_id')
        if 'attr_' in field_cols :
          field_cols.remove('attr_')
          field_cols.extend(flattenedColumns)

        if 'attr_' in tag_cols :
          tag_cols.remove('attr_')
          tag_cols.extend(flattenedColumns)
        lineProtocolTimestampedDF = toLineProtocol(storageName, "TimestampedEventSeries", eventsTimestampedEntryDFTF3, tag_cols, field_cols, "_timestamp")
        storagePath = lineProtocolPath + "/" + storageName
        lineProtocolTimestampedDF.coalesce(12).write.format("text").option("header", "false").mode("append").save(storagePath)
    if len(eventsTimestampedEmptyAEntryTF1.head(1)) != 0: 
      field_cols = fields.split(",")
      tag_cols = tags.split(",")
      empty_Attr_tag_cols = tags.split(",")   
      if 'attr_' in empty_Attr_tag_cols :
        empty_Attr_tag_cols.remove('attr_')

      empty_Attr_field_cols = fields.split(",")
      if 'attr_' in empty_Attr_field_cols :   
        empty_Attr_field_cols.remove('attr_')
      try :
        eventsTimestampedEmptyAEntryTF2 = eventsTimestampedEmptyAEntryTF1.select(parseId("s").alias("series_id"),col("es").alias("source_id"),parseId("_id").alias("instance_id"),col("d").alias("data"),col("t").alias("_timestamp"),"*").drop("es","s","_id","d","t")
      except :
        eventsTimestampedEmptyAEntryTF2 = eventsTimestampedEmptyAEntryTF1.select(parseId("s").alias("series_id"),parseId("_id").alias("instance_id"),col("d").alias("data"),col("t").alias("_timestamp"),"*").drop("s","_id","d","t")
        empty_Attr_tag_cols.remove('source_id')
      lineProtocolEmptyTimestampedDF = toLineProtocol(storageName, "TimestampedEventSeries", eventsTimestampedEmptyAEntryTF2, empty_Attr_tag_cols, empty_Attr_field_cols, "_timestamp")
      storagePath = lineProtocolPath + "/" + storageName
      lineProtocolEmptyTimestampedDF.coalesce(12).write.format("text").option("header", "false").mode("append").save(storagePath)